In [ ]:
import numpy as np
import rasterio as rio
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
from pyproj import CRS, Transformer
from rasterio.plot import show
import xarray as xr
import pandas as pd
from pathlib import Path

In [ ]:
lst_example = rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/BatchExport_LST/2000/GEMLST_MODIS_20000101.tif')


In [ ]:
# ERA5 Data: 
imfolder = "./GL500m/t2m/"
imfiles = sorted(Path(imfolder).glob("*.nc"))

# CORRUPTED FILES TO EXCLUDE
#	t2m_elvcorr_2001_d201 until t2m_elvcorr_2001_d210
#	t2m_elvcorr_2001_d339 until t2m_elvcorr_2001_d349
#	t2m_elvcorr_2019_d160

# markers (adjust if you need case-insensitive match or different substrings)
start_marker = "t2m_elvcorr_2000_d001.nc" # OBS D11
end_marker = "t2m_elvcorr_2000_d366.nc"   # OBS LEAP YEARS

# find first index containing the start marker and last index containing the end marker
start_idx = next((i for i, p in enumerate(imfiles) if start_marker in p.name), None)
end_idx = next((i for i, p in enumerate(imfiles) if end_marker in p.name), None)

if start_idx is None:
    raise FileNotFoundError(f"No file containing '{start_marker}' found under {imfolder}")
if end_idx is None:
    raise FileNotFoundError(f"No file containing '{end_marker}' found under {imfolder}")
if end_idx < start_idx:
    raise ValueError(f"End file '{end_marker}' appears before start file '{start_marker}' in sorted file order")

# slice inclusive range
imfiles = imfiles[start_idx:end_idx + 1]

In [ ]:
for imfile in imfiles:

### REPROJECTION ###

    # Load ERA5 data
    with xr.open_dataset(imfile) as ds:
        t2m = ds['t2m']
        lat = ds['Y']
        lon = ds['X']
    t2m = np.squeeze(t2m.values) - 273.15  # Convert from Kelvin to Celsius    

    crs_wgs84 = CRS.from_epsg(4326)  # WGS84
    crs_3413 = CRS.from_epsg(3413)   # EPSG:3413
    
    transformer = Transformer.from_crs(crs_wgs84, crs_3413, always_xy=True)
    x_proj, y_proj = transformer.transform(lon.values, lat.values)

    # Ensure ERA5 data match the spatial extent and resolution of the MODIS LST data
    lst_data = lst_example.read(1)
    lst_transform = lst_example.transform
    lst_crs = lst_example.crs
    lst_bounds = lst_example.bounds
    lst_width = lst_example.width
    lst_height = lst_example.height

    ## Create an array to hold the reprojected ERA5 data
    reprojected_t2m = np.empty((lst_height, lst_width), dtype=np.float32)

    # Reproject the ERA5 data to match the MODIS grid
    reproject(
        source=t2m,  # Remove any singleton dimensions
        destination=reprojected_t2m,
        src_transform=from_bounds(x_proj.min(), y_proj.min(), x_proj.max(), y_proj.max(), t2m.shape[1], t2m.shape[0]),
        src_crs=crs_3413,
        dst_transform=lst_transform,
        dst_crs=lst_crs,
        resampling=Resampling.bilinear,
        src_nodata=np.nan,
        dst_nodata=np.nan
    )

    
### SAVE FILE ### 
    output_file = './GL1000m_reproj/t2m_1000m_FILENAME.tif'
    output_file = output_file.replace("FILENAME", imfile.stem.split('_',2)[2])
    with rio.open(output_file, 'w', driver='GTiff', height=lst_height, width=lst_width, count=1, dtype=np.float32, crs=lst_crs, transform=lst_transform) as dst:
        dst.write(reprojected_t2m, 1)

### PRINTS###
    # Every round
    print(f'File {imfile} reprojected')

# Confirm loop completion
print('ALL FILES DONE') 


getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d001.nc: Operation not supported
getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d001.nc: Operation not supported


File GL500m/t2m/t2m_elvcorr_2000_d001.nc reprojected


getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d002.nc: Operation not supported
getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d002.nc: Operation not supported


File GL500m/t2m/t2m_elvcorr_2000_d002.nc reprojected


getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d003.nc: Operation not supported
getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d003.nc: Operation not supported


...

getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d194.nc: Operation not supported
getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d194.nc: Operation not supported


File GL500m/t2m/t2m_elvcorr_2000_d194.nc reprojected


getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d195.nc: Operation not supported
getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d195.nc: Operation not supported


File GL500m/t2m/t2m_elvcorr_2000_d195.nc reprojected


getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d196.nc: Operation not supported
getfattr: /home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL500m/t2m/t2m_elvcorr_2000_d196.nc: Operation not supported
